In [0]:
from pyspark.sql import functions as F

##### 1.1 预先铺设数据管道


In [0]:
VOLUME_BASE = "/Volumes/workspace/default/olist_files"

##### 1.2 导入订单表


In [0]:
orders_df = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(f"{VOLUME_BASE}/olist_orders_dataset.csv")

display(orders_df.limit(5))

##### 1.3 导入用户主表


In [0]:
customers_df = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(f"{VOLUME_BASE}/olist_customers_dataset.csv")

display(customers_df.limit(5))

##### 1.4 导入商品表


In [0]:
products_df = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(f"{VOLUME_BASE}/olist_products_dataset.csv")

display(products_df.limit(5))

##### 1.4 导入订单详情表


In [0]:
items_df = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(f"{VOLUME_BASE}/olist_order_items_dataset.csv")

display(items_df.limit(5))

###### 2. 矩阵式注册临时视图（构建内存虚拟数仓）

In [0]:
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")
products_df.createOrReplaceTempView("products")
items_df.createOrReplaceTempView("items")
print("✅ 【数据准备就绪】核心业务表已成功映射为 SQL 临时视图！")

##### 3. 编写重工业 SQL 复现 BQ 核心商业指标

In [0]:
BQ_metrics_df = spark.sql("""
    SELECT
        round(SUM(oi.price),2) AS GMV,
        count(DISTINCT(o.order_id)) AS total_orders,
        count(DISTINCT(c.customer_id)) AS total_customers,

        -- 核心效率：客单价 (AOV)
        ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
    FROM  items oi
    INNER JOIN orders o ON o.order_id = oi.order_id
    INNER JOIN customers c ON c.customer_id = o.customer_id
    INNER JOIN products p ON p.product_id = oi.product_id                      
""")

print("\n🚀 【真实指标计算完成】还原真实 Olist 级联后的商业指标：")
display(BQ_metrics_df)